# Comparative Analysis of Learning Paradigms for Text Classification

This notebook combines the project scripts into one notebook while keeping each script in its own section. Run sections independently when needed because some cells train models or run large LLM inference experiments.


## Part 1 - Data Exploration

Source script: `Data Exploration.py`

Dataset inspection, class distribution, text-length plots, and preprocessing decisions.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer


In [ ]:
# Load the dair-ai/emotion dataset from Hugging Face Hub
dataset = load_dataset("dair-ai/emotion")

# High-level info: splits, feature names, row counts
print(dataset)

# One raw example from each split so we understand the schema
for split in ["train", "validation", "test"]:
    print(f"\n--- {split.upper()} SAMPLE ---")
    print(dataset[split][0])


In [ ]:
# Human-readable names for the six integer label IDs
label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]

# Total examples per split
for split in ["train", "validation", "test"]:
    print(f"{split}: {len(dataset[split])} examples")

# Per-class counts for every split
for split in ["train", "validation", "test"]:
    labels = dataset[split]["label"]
    counts = Counter(labels)
    print(f"\n{split.upper()}:")
    for label_id, count in sorted(counts.items()):
        print(f"  {label_names[label_id]:<10} : {count}")


In [ ]:
# Character-level length statistics (min / max / average) per split
for split in ["train", "validation", "test"]:
    lengths = [len(text) for text in dataset[split]["text"]]
    avg = sum(lengths) / len(lengths)
    print(f"  {split} — min: {min(lengths)}, max: {max(lengths)}, avg: {avg:.1f}")


In [ ]:
# Bar chart: how many examples belong to each emotion in each split
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, split in enumerate(["train", "validation", "test"]):
    labels = dataset[split]["label"]
    counts = Counter(labels)
    names  = [label_names[k] for k in sorted(counts)]   # ordered by label ID
    values = [counts[k]       for k in sorted(counts)]

    axes[i].bar(names, values, color="steelblue")
    axes[i].set_title(f"{split.capitalize()} — Class Distribution")
    axes[i].set_xlabel("Emotion")
    axes[i].set_ylabel("Count")
    axes[i].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()


In [ ]:
# Histogram: distribution of text lengths in characters per split
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, split in enumerate(["train", "validation", "test"]):
    lengths = [len(text) for text in dataset[split]["text"]]
    axes[i].hist(lengths, bins=40, color="coral", edgecolor="white")
    axes[i].set_title(f"{split.capitalize()} — Text Length")
    axes[i].set_xlabel("Characters")
    axes[i].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("text_length_distribution.png", dpi=150)
plt.show()


In [ ]:
def simple_tokenize(text):
    # Lowercase + whitespace split; no stemming or punctuation removal
    # (punctuation carries emotional signal e.g. "fine. whatever.")
    return text.lower().split()

# Count every token across the entire training set
word_freq = Counter()
for text in dataset["train"]["text"]:
    word_freq.update(simple_tokenize(text))

PAD_TOKEN = "<PAD>"  # padding index = 0
UNK_TOKEN = "<UNK>"  # out-of-vocabulary fallback index = 1

# Words appearing only once are likely noise; frequency threshold >= 2
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word, freq in word_freq.most_common():
    if freq >= 2:
        vocab[word] = len(vocab)

print(f"Vocabulary size : {len(vocab)}")


In [ ]:
def encode_rnn(text, vocab, max_len=50):
    tokens = simple_tokenize(text)[:max_len]          # truncate to max_len
    ids    = [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens]  # OOV → UNK
    ids   += [vocab[PAD_TOKEN]] * (max_len - len(ids))         # right-pad
    return ids

# Show encoding of the first training example
sample_text = dataset["train"][0]["text"]
print("Text   :", sample_text)
print("Encoded:", encode_rnn(sample_text, vocab))


In [ ]:
# Load the pre-trained DistilBERT tokenizer (WordPiece, 30k-token vocab)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample_text = dataset["train"][0]["text"]
encoded = tokenizer(
    sample_text,
    truncation=True,       # clip sequences longer than max_length
    padding="max_length",  # pad shorter sequences to max_length
    max_length=64,         # 64 tokens covers all examples; 512 is overkill
    return_tensors="pt"
)

print("Text          :", sample_text)
print("Input IDs     :", encoded["input_ids"])
print("Attention Mask:", encoded["attention_mask"])  # 1 = real token, 0 = pad
print("Shape         :", encoded["input_ids"].shape)


## Preprocessing Decisions

| Choice | Rationale |
|--------|-----------|
| Lowercasing (RNN only) | Reduces vocab size. BERT handles casing internally via WordPiece. |
| Frequency threshold ≥ 2 (RNN) | Words appearing once are likely noise; dropping them shrinks the embedding table. |
| `max_length = 50` tokens (RNN) | 99%+ of tweets fit within 50 whitespace-split tokens. |
| `max_length = 64` (BERT) | Covers all examples with room to spare; BERT's default 512 is overkill. |
| No punctuation stripping | Punctuation carries emotional signal ("fine. whatever." vs "fine whatever"). |
| No stop-word removal | Emotion often lives in function words ("I do love", "I don't care"). |


## Part 2 - Recurrent Baseline

Source script: `Recurrent-Baseline.py`

From-scratch GRU/LSTM emotion classifier using random embeddings and labeled data only.


In [7]:
# %% [Cell 1] Imports and reproducibility setup
import os
import random
import re
import time
from collections import Counter

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from datasets import Dataset as HFDataset
from datasets import DatasetDict, load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset as TorchDataset


# A fixed seed makes the random embedding initialization and training order reproducible.
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [8]:
# %% [Cell 2] Dataset loading and label setup
# Required bullet: Train only from the labeled data.
# We use the same labeled Hugging Face emotion dataset as the rest of the project.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]
NUM_LABELS = len(label_names)


def load_emotion_dataset():
    """Load cached Arrow splits first, then fall back to Hugging Face if needed."""
    cache_root = os.path.join(
        os.path.expanduser("~"),
        ".cache",
        "huggingface",
        "datasets",
        "dair-ai___emotion",
        "split",
        "0.0.0",
    )
    if os.path.isdir(cache_root):
        for revision in os.listdir(cache_root):
            revision_dir = os.path.join(cache_root, revision)
            split_paths = {
                "train": os.path.join(revision_dir, "emotion-train.arrow"),
                "validation": os.path.join(revision_dir, "emotion-validation.arrow"),
                "test": os.path.join(revision_dir, "emotion-test.arrow"),
            }
            if all(os.path.exists(path) for path in split_paths.values()):
                print(f"Loading cached dair-ai/emotion Arrow files from: {revision_dir}")
                return DatasetDict(
                    {
                        split: HFDataset.from_file(path)
                        for split, path in split_paths.items()
                    }
                )

    print("Cached Arrow files not found; loading dair-ai/emotion through Hugging Face.")
    return load_dataset("dair-ai/emotion")


dataset = load_emotion_dataset()


def env_int(name, default):
    """Read integer hyperparameters from environment variables when provided."""
    value = os.getenv(name)
    if value is None:
        return default
    try:
        return int(value)
    except ValueError:
        print(f"Ignoring invalid {name}={value!r}; using {default}.")
        return default


# FAST_DEV_RUN is only for quick debugging. The default uses the full dataset.
FAST_DEV_RUN = os.getenv("FAST_DEV_RUN", "").lower() in {"1", "true", "yes"}
BATCH_SIZE = env_int("BATCH_SIZE", 16 if FAST_DEV_RUN else 64)
EPOCHS = env_int("EPOCHS", 2 if FAST_DEV_RUN else 10)
MAX_LEN = env_int("MAX_LEN", 50)
TRAIN_SAMPLE_LIMIT = env_int("TRAIN_SAMPLE_LIMIT", 256 if FAST_DEV_RUN else 0)
VAL_SAMPLE_LIMIT = env_int("VAL_SAMPLE_LIMIT", 128 if FAST_DEV_RUN else 0)
TEST_SAMPLE_LIMIT = env_int("TEST_SAMPLE_LIMIT", 128 if FAST_DEV_RUN else 0)


Using device: cpu
Loading cached dair-ai/emotion Arrow files from: C:\Users\mazen\.cache\huggingface\datasets\dair-ai___emotion\split\0.0.0\cab853a1dbdf4c42c2b3ef2173804746df8825fe


In [9]:
# %% [Cell 3] Vocabulary building from the training split only
# Required bullet: Randomly initialized embedding layer.
# The embedding matrix is random, so this vocabulary is built from labeled train text only.
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1


def simple_tokenize(text):
    """Lowercase and tokenize words/punctuation without using pretrained tokenizers."""
    return re.findall(r"[a-z]+(?:'[a-z]+)?|[0-9]+|[^\w\s]", text.lower())


def limit_split(split_name, limit):
    """Optionally shorten a split for FAST_DEV_RUN while keeping full-data defaults."""
    split_dataset = dataset[split_name]
    if limit > 0:
        split_dataset = split_dataset.select(range(min(limit, len(split_dataset))))
    return split_dataset


train_split = limit_split("train", TRAIN_SAMPLE_LIMIT)
val_split = limit_split("validation", VAL_SAMPLE_LIMIT)
test_split = limit_split("test", TEST_SAMPLE_LIMIT)

word_freq = Counter()
for text in train_split["text"]:
    word_freq.update(simple_tokenize(text))

vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
for token, freq in word_freq.most_common():
    # Keeping words that appear at least twice reduces noise and keeps the model compact.
    if freq >= 2:
        vocab[token] = len(vocab)

print(f"Vocabulary size: {len(vocab):,}")
print(f"Train/Val/Test examples: {len(train_split):,}/{len(val_split):,}/{len(test_split):,}")


Vocabulary size: 510
Train/Val/Test examples: 256/128/128


In [10]:
# %% [Cell 4] Encoding text and creating PyTorch datasets
# Required bullet: Train on the full training set with an appropriate batch size.
# Each text becomes a fixed-length integer sequence plus its real length for the RNN.
def encode_text(text, max_len=MAX_LEN):
    tokens = simple_tokenize(text)[:max_len]
    token_ids = [vocab.get(token, UNK_IDX) for token in tokens]
    length = max(1, len(token_ids))
    token_ids += [PAD_IDX] * (max_len - len(token_ids))
    return token_ids, length


class EmotionRnnDataset(TorchDataset):
    def __init__(self, split_dataset):
        self.texts = list(split_dataset["text"])
        self.labels = torch.tensor(list(split_dataset["label"]), dtype=torch.long)
        encoded = [encode_text(text) for text in self.texts]
        self.input_ids = torch.tensor([item[0] for item in encoded], dtype=torch.long)
        self.lengths = torch.tensor([item[1] for item in encoded], dtype=torch.long)

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, index):
        return {
            "input_ids": self.input_ids[index],
            "lengths": self.lengths[index],
            "labels": self.labels[index],
        }


train_loader = DataLoader(EmotionRnnDataset(train_split), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(EmotionRnnDataset(val_split), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(EmotionRnnDataset(test_split), batch_size=BATCH_SIZE, shuffle=False)


In [11]:
# %% [Cell 5] LSTM/GRU classifier architecture
# Required bullet: Build an LSTM or GRU classifier with random embeddings, recurrent layers,
# and a final dense classification head with softmax.
class RecurrentEmotionClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_labels,
        rnn_type="GRU",
        embedding_dim=128,
        hidden_dim=128,
        num_layers=2,
        dropout=0.3,
        bidirectional=True,
    ):
        super().__init__()
        if rnn_type not in {"GRU", "LSTM"}:
            raise ValueError("rnn_type must be 'GRU' or 'LSTM'")

        self.rnn_type = rnn_type
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        # Random embedding layer: no pretrained vectors are loaded.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)

        rnn_class = nn.GRU if rnn_type == "GRU" else nn.LSTM
        self.rnn = rnn_class(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        # Final dense classification head. Softmax is exposed for inference probabilities.
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * self.num_directions, num_labels)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, input_ids, lengths, return_probs=False):
        embedded = self.embedding(input_ids)

        # Packing tells the RNN to ignore padding tokens when building the sentence vector.
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, hidden = self.rnn(packed)

        # LSTM returns (hidden_state, cell_state); GRU returns hidden_state only.
        if self.rnn_type == "LSTM":
            hidden = hidden[0]

        if self.bidirectional:
            sentence_vector = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            sentence_vector = hidden[-1]

        logits = self.classifier(self.dropout(sentence_vector))
        if return_probs:
            return self.softmax(logits)
        return logits


In [12]:
# %% [Cell 6] Training and validation helpers
# Required bullet: Train with an appropriate optimizer, learning rate, and batch size.
# CrossEntropyLoss expects raw logits, so softmax is not applied during training.
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.set_grad_enabled(is_training):
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["lengths"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(
        all_labels,
        all_preds,
        labels=range(NUM_LABELS),
        average="macro",
        zero_division=0,
    )
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy, macro_f1


def train_model(rnn_type):
    """Train one recurrent model and keep the best validation checkpoint."""
    print(f"\nTraining {rnn_type} baseline...\n")

    model = RecurrentEmotionClassifier(
        vocab_size=len(vocab),
        num_labels=NUM_LABELS,
        rnn_type=rnn_type,
    ).to(device)

    print(f"{rnn_type} total parameters    : {sum(p.numel() for p in model.parameters()):,}")
    print(f"{rnn_type} trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "train_macro_f1": [],
        "val_macro_f1": [],
    }
    best_val_macro_f1 = -1.0
    best_state = None
    train_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc, train_macro_f1 = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_macro_f1 = run_epoch(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_macro_f1"].append(train_macro_f1)
        history["val_macro_f1"].append(val_macro_f1)

        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

        print(
            f"{rnn_type} Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}"
        )

    training_time = time.time() - train_start
    if best_state is not None:
        model.load_state_dict(best_state)

    return {
        "name": rnn_type,
        "model": model,
        "history": history,
        "best_val_macro_f1": best_val_macro_f1,
        "training_time": training_time,
    }


In [13]:
# %% [Cell 7] Plot training and validation loss curves across epochs
# Required bullet: Plot training and validation loss curves across epochs.
def plot_loss_curves(results):
    epochs_range = range(1, EPOCHS + 1)
    plt.figure(figsize=(10, 6))

    for result in results:
        name = result["name"]
        history = result["history"]
        plt.plot(epochs_range, history["train_loss"], marker="o", label=f"{name} Train Loss")
        plt.plot(epochs_range, history["val_loss"], marker="s", linestyle="--", label=f"{name} Val Loss")

    plt.title("Training and Validation Loss - Recurrent Baselines")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-Entropy Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig("rnn_training_curves.png", dpi=150)
    plt.close()


In [14]:
# %% [Cell 8] Test evaluation and inference timing
# Required bullet: Evaluate on the test set and report accuracy, macro F1, and per-class F1.
# Required bullet: Measure and report inference time on the test set.
def evaluate_on_test(model):
    model.eval()
    all_preds = []
    all_labels = []

    infer_start = time.time()
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["lengths"].to(device)
            labels = batch["labels"].to(device)

            # The model exposes softmax probabilities for inference, then argmax picks a class.
            probs = model(input_ids, lengths, return_probs=True)
            preds = probs.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    inference_time = time.time() - infer_start
    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(
        all_labels,
        all_preds,
        labels=range(NUM_LABELS),
        average="macro",
        zero_division=0,
    )
    per_class_f1 = f1_score(
        all_labels,
        all_preds,
        labels=range(NUM_LABELS),
        average=None,
        zero_division=0,
    )

    return {
        "labels": all_labels,
        "preds": all_preds,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "per_class_f1": per_class_f1,
        "inference_time": inference_time,
    }


In [15]:
# %% [Cell 9] Confusion matrix display for the test set
# Required bullet: Display a confusion matrix on the test set.
def plot_confusion_matrix(labels, preds, model_name):
    cm = confusion_matrix(labels, preds, labels=range(NUM_LABELS))

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_names,
        yticklabels=label_names,
    )
    plt.title(f"Confusion Matrix - {model_name} Recurrent Baseline")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig("rnn_confusion_matrix.png", dpi=150)
    plt.close()


In [16]:
# %% [Cell 10] Main experiment runner
# Required bullet: Measure and report total training time.
experiment_start = time.time()

# Train both candidates under identical settings, then select by validation macro F1.
results = [train_model("GRU"), train_model("LSTM")]
plot_loss_curves(results)

best_result = max(results, key=lambda item: item["best_val_macro_f1"])
best_model = best_result["model"]
test_metrics = evaluate_on_test(best_model)
plot_confusion_matrix(test_metrics["labels"], test_metrics["preds"], best_result["name"])

total_training_time = sum(result["training_time"] for result in results)
total_experiment_time = time.time() - experiment_start

print("\nBest recurrent model")
print("--------------------")
for result in results:
    print(
        f"{result['name']}: best validation macro F1 = {result['best_val_macro_f1']:.4f}, "
        f"training time = {result['training_time']:.1f}s"
    )
print(f"Selected model: {best_result['name']}")

print("\nTest set metrics")
print("----------------")
print(f"Accuracy       : {test_metrics['accuracy']:.4f}")
print(f"Macro F1       : {test_metrics['macro_f1']:.4f}")
print(f"Inference time : {test_metrics['inference_time']:.2f}s")
print(f"Training time  : {total_training_time:.1f}s")
print(f"Experiment time: {total_experiment_time:.1f}s")

print("\nPer-class F1")
print("------------")
for name, score in zip(label_names, test_metrics["per_class_f1"]):
    print(f"{name:<10}: {score:.4f}")

print("\nDetailed classification report")
print("------------------------------")
print(
    classification_report(
        test_metrics["labels"],
        test_metrics["preds"],
        labels=range(NUM_LABELS),
        target_names=label_names,
        zero_division=0,
    )
)

print("Saved plots: rnn_training_curves.png, rnn_confusion_matrix.png")



Training GRU baseline...

GRU total parameters    : 561,414
GRU trainable parameters: 561,414


GRU Epoch 01/2 | Train Loss: 1.6429, Train Macro F1: 0.1547 | Val Loss: 1.6191, Val Macro F1: 0.0953


GRU Epoch 02/2 | Train Loss: 1.4953, Train Macro F1: 0.1983 | Val Loss: 1.6267, Val Macro F1: 0.0908

Training LSTM baseline...

LSTM total parameters    : 726,278
LSTM trainable parameters: 726,278


LSTM Epoch 01/2 | Train Loss: 1.6803, Train Macro F1: 0.0864 | Val Loss: 1.6623, Val Macro F1: 0.0716


LSTM Epoch 02/2 | Train Loss: 1.5589, Train Macro F1: 0.1046 | Val Loss: 1.5872, Val Macro F1: 0.0725



Best recurrent model
--------------------
GRU: best validation macro F1 = 0.0953, training time = 7.0s
LSTM: best validation macro F1 = 0.0725, training time = 6.9s
Selected model: GRU

Test set metrics
----------------
Accuracy       : 0.3125
Macro F1       : 0.1026
Inference time : 0.26s
Training time  : 13.9s
Experiment time: 14.9s

Per-class F1
------------
sadness   : 0.1509
joy       : 0.4645
love      : 0.0000
anger     : 0.0000
fear      : 0.0000
surprise  : 0.0000

Detailed classification report
------------------------------
              precision    recall  f1-score   support

     sadness       0.36      0.10      0.15        42
         joy       0.31      0.92      0.46        39
        love       0.00      0.00      0.00         7
       anger       0.00      0.00      0.00        20
        fear       0.00      0.00      0.00        17
    surprise       0.00      0.00      0.00         3

    accuracy                           0.31       128
   macro avg       0.11 

## Part 3 - Transformer Fine-Tuning

Source script: `Finetunning-Transformer.py`

BERT fine-tuning baseline with training curves, test metrics, and confusion matrix.


In [17]:
# %% [Cell 1] Imports
import os
import time
import torch
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report


In [18]:
# %% [Cell 2] Device setup and dataset loading
# Use GPU if available; BERT fine-tuning is very slow on CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]
NUM_LABELS  = len(label_names)


def env_int(name, default):
    value = os.getenv(name)
    if value is None:
        return default
    try:
        return int(value)
    except ValueError:
        print(f"Ignoring invalid {name}={value!r}; using {default}.")
        return default


FAST_DEV_RUN = os.getenv("FAST_DEV_RUN", "").lower() in {"1", "true", "yes"}
BATCH_SIZE = env_int("BATCH_SIZE", 8 if FAST_DEV_RUN else 32)
TRAIN_SAMPLE_LIMIT = env_int("TRAIN_SAMPLE_LIMIT", 64 if FAST_DEV_RUN else 0)
VAL_SAMPLE_LIMIT = env_int("VAL_SAMPLE_LIMIT", 32 if FAST_DEV_RUN else 0)
TEST_SAMPLE_LIMIT = env_int("TEST_SAMPLE_LIMIT", 32 if FAST_DEV_RUN else 0)

# Load the same emotion dataset used in Part 1
dataset = load_dataset("dair-ai/emotion")


Using device: cpu


In [19]:
# %% [Cell 3] Tokenizer and PyTorch Dataset class
# bert-base-uncased uses WordPiece tokenization with a 30k-token vocabulary
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

class EmotionDataset(Dataset):
    def __init__(self, split, sample_limit=0):
        split_dataset = dataset[split]
        if sample_limit > 0:
            split_dataset = split_dataset.select(range(min(sample_limit, len(split_dataset))))

        self.texts = list(split_dataset["text"])
        self.encodings = tokenizer(
            self.texts,
            truncation=True,
            padding="max_length",
            max_length=64,
            return_tensors="pt"
        )
        self.labels = torch.tensor(list(split_dataset["label"]), dtype=torch.long)

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx]
        }


In [20]:
# %% [Cell 4] DataLoaders — batch and shuffle the three splits
# Shuffle training data each epoch to prevent ordering bias
train_loader = DataLoader(EmotionDataset("train", TRAIN_SAMPLE_LIMIT), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(EmotionDataset("validation", VAL_SAMPLE_LIMIT), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(EmotionDataset("test", TEST_SAMPLE_LIMIT), batch_size=BATCH_SIZE, shuffle=False)


In [21]:
# %% [Cell 5] Load pre-trained BERT and attach a classification head
# AutoModelForSequenceClassification replaces BERT's MLM head with a linear
# classifier that outputs one logit per emotion class
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
).to(device)

# Sanity-check model size: all ~110M parameters are trainable (full fine-tuning)
print(f"Total parameters    : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2449.48it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters    : 109,486,854
Trainable parameters: 109,486,854


In [22]:
# %% [Cell 6] Optimizer and learning-rate scheduler
EPOCHS = env_int("EPOCHS", 1 if FAST_DEV_RUN else 3)
LR     = 2e-5  # standard BERT fine-tuning range: 1e-5 to 5e-5

# AdamW (Adam + decoupled weight decay) is the standard choice for BERT
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS

# Linear warmup for the first 10% of steps, then linear decay to 0
# Warmup prevents large gradient updates that could destroy pre-trained weights
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)


In [23]:
# %% [Cell 7] Training / evaluation loop function
def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    # Disable gradient computation during validation to save memory and time
    with torch.set_grad_enabled(training):
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            # Forward pass; HuggingFace models compute cross-entropy loss internally
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss    = outputs.loss

            if training:
                optimizer.zero_grad()
                loss.backward()
                # Clip gradients to prevent exploding gradients in deep transformers
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()
            # Predicted class = index of the highest logit
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


In [24]:
# %% [Cell 8] Run training loop and record metrics per epoch
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

print("\nTraining...\n")
train_start = time.time()

for epoch in range(EPOCHS):
    # Full pass over training data (with gradient updates)
    t_loss, t_acc = run_epoch(train_loader, training=True)
    # Full pass over validation data (no gradient updates)
    v_loss, v_acc = run_epoch(val_loader,   training=False)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    train_accs.append(t_acc)
    val_accs.append(v_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} — "
          f"Train Loss: {t_loss:.4f}, Train Acc: {t_acc:.4f} | "
          f"Val Loss: {v_loss:.4f}, Val Acc: {v_acc:.4f}")

train_time = time.time() - train_start
print(f"\nTotal training time: {train_time:.1f}s")



Training...



Epoch 1/1 — Train Loss: 1.7126, Train Acc: 0.3125 | Val Loss: 1.5797, Val Acc: 0.3125

Total training time: 27.0s


In [25]:
# %% [Cell 9] Plot loss and accuracy curves
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(12, 4))

# Loss subplot — convergence check; val loss rising signals overfitting
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label="Train Loss")
plt.plot(epochs_range, val_losses,   label="Val Loss")
plt.title("Loss Curves — BERT Fine-Tuning")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Accuracy subplot — visual confirmation that val acc tracks train acc
plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_accs, label="Train Acc")
plt.plot(epochs_range, val_accs,   label="Val Acc")
plt.title("Accuracy Curves — BERT Fine-Tuning")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.savefig("bert_training_curves.png", dpi=150)
plt.close()


In [26]:
# %% [Cell 10] Evaluate on the held-out test set
model.eval()
all_preds, all_labels = [], []

infer_start = time.time()

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        # No labels passed here — we only need logits for inference
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = outputs.logits.argmax(dim=-1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

infer_time = time.time() - infer_start

# Macro F1 treats each class equally regardless of support — important for
# imbalanced classes like "surprise" and "love"
accuracy = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, labels=range(NUM_LABELS), average="macro", zero_division=0)

print(f"\nTest Accuracy : {accuracy:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Inference Time: {infer_time:.2f}s\n")
print(classification_report(
    all_labels,
    all_preds,
    labels=range(NUM_LABELS),
    target_names=label_names,
    zero_division=0
))



Test Accuracy : 0.2500
Macro F1      : 0.0993
Inference Time: 2.32s

              precision    recall  f1-score   support

     sadness       0.18      0.18      0.18        11
         joy       0.29      0.75      0.41         8
        love       0.00      0.00      0.00         1
       anger       0.00      0.00      0.00         6
        fear       0.00      0.00      0.00         6
    surprise       0.00      0.00      0.00         0

    accuracy                           0.25        32
   macro avg       0.08      0.16      0.10        32
weighted avg       0.13      0.25      0.17        32



In [27]:
# %% [Cell 11] Confusion matrix — per-class error analysis
cm = confusion_matrix(all_labels, all_preds, labels=range(NUM_LABELS))

plt.figure(figsize=(8, 6))
# annot=True prints raw counts in each cell; fmt="d" keeps them as integers
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names)
plt.title("Confusion Matrix — BERT Fine-Tuned")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("bert_confusion_matrix.png", dpi=150)
plt.close()


In [ ]:
# Store Part 3 metrics under stable names for Part 5 & 6 comparisons
bert_full_acc        = accuracy
bert_full_f1         = macro_f1
bert_full_train_time = train_time
bert_full_infer_time = infer_time

print(f"BERT Test Accuracy  : {bert_full_acc:.4f}")
print(f"BERT Macro F1       : {bert_full_f1:.4f}")
print(f"Training time       : {bert_full_train_time:.1f}s")
print(f"Inference time      : {bert_full_infer_time:.2f}s")


## Part 3 — Model Analysis

### Parameters
`bert-base-uncased` has **~110 million parameters**, all of which are updated during fine-tuning
(full fine-tuning — no layers are frozen).

### Pretraining Objective
BERT was pretrained on BookCorpus and English Wikipedia using two self-supervised tasks:
1. **Masked Language Modelling (MLM)** — 15% of tokens are masked; the model predicts them from bidirectional context.
2. **Next Sentence Prediction (NSP)** — the model predicts whether two sentences are consecutive.

### Why Pretraining Helps vs. the From-Scratch RNN
When we fine-tune BERT, we are *not* teaching it what language is — it already knows.
We are only redirecting its rich, bidirectional contextual representations toward our six emotion labels.
The from-scratch RNN (Part 2) must learn language structure, word meaning, and the classification task
simultaneously from only 16,000 short tweets, which is why BERT consistently outperforms it —
especially on minority classes such as **love** and **surprise** with low training-set support.


## Part 4 - LLM Prompting

Source script: `LLM-Prompting.py`

Instruction-tuned LLM zero-shot, few-shot, and chain-of-thought prompting without gradient updates.


In [29]:
# %% [Cell 1] Imports and experiment configuration
import json
import os
import re
import time
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import Dataset as HFDataset
from datasets import DatasetDict, load_dataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from transformers import BitsAndBytesConfig
except ImportError:
    BitsAndBytesConfig = None


# This notebook-style script evaluates an instruction-tuned LLM with no gradient updates.
# Default model for the assignment: microsoft/Phi-3-mini-4k-instruct.
# On a CPU-only machine, use FAST_DEV_RUN=1 and/or set MODEL_ID to a smaller local model.
MODEL_ID = os.getenv("MODEL_ID", "microsoft/Phi-3-mini-4k-instruct")
FAST_DEV_RUN = os.getenv("FAST_DEV_RUN", "").lower() in {"1", "true", "yes"}
TEST_SAMPLE_LIMIT = int(os.getenv("TEST_SAMPLE_LIMIT", "24" if FAST_DEV_RUN else "0"))
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "96"))
COT_MAX_NEW_TOKENS = int(os.getenv("COT_MAX_NEW_TOKENS", "160"))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_4BIT = (
    DEVICE.type == "cuda"
    and BitsAndBytesConfig is not None
    and os.getenv("DISABLE_4BIT", "").lower() not in {"1", "true", "yes"}
)

OUTPUT_DIR = Path("llm_prompting_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]
label_to_id = {label: index for index, label in enumerate(label_names)}

print(f"Using device: {DEVICE}")
print(f"Model id: {MODEL_ID}")
print(f"4-bit quantization enabled: {USE_4BIT}")


Using device: cpu
Model id: HuggingFaceTB/SmolLM2-135M-Instruct
4-bit quantization enabled: False


In [30]:
# %% [Cell 2] Load the labeled emotion dataset without train-time updates
def load_emotion_dataset():
    """Load cached Arrow splits first, then fall back to Hugging Face if needed."""
    cache_root = Path.home() / ".cache" / "huggingface" / "datasets" / "dair-ai___emotion" / "split" / "0.0.0"
    if cache_root.exists():
        for revision_dir in cache_root.iterdir():
            split_paths = {
                "train": revision_dir / "emotion-train.arrow",
                "validation": revision_dir / "emotion-validation.arrow",
                "test": revision_dir / "emotion-test.arrow",
            }
            if all(path.exists() for path in split_paths.values()):
                print(f"Loading cached dair-ai/emotion Arrow files from: {revision_dir}")
                return DatasetDict(
                    {split: HFDataset.from_file(str(path)) for split, path in split_paths.items()}
                )

    print("Cached Arrow files not found; loading dair-ai/emotion through Hugging Face.")
    return load_dataset("dair-ai/emotion")


dataset = load_emotion_dataset()
train_split = dataset["train"]
test_split = dataset["test"]
if TEST_SAMPLE_LIMIT > 0:
    test_split = test_split.select(range(min(TEST_SAMPLE_LIMIT, len(test_split))))

print(f"Train examples available for few-shot prompts: {len(train_split):,}")
print(f"Test examples evaluated: {len(test_split):,}")


Loading cached dair-ai/emotion Arrow files from: C:\Users\mazen\.cache\huggingface\datasets\dair-ai___emotion\split\0.0.0\cab853a1dbdf4c42c2b3ef2173804746df8825fe
Train examples available for few-shot prompts: 16,000
Test examples evaluated: 24


In [31]:
# %% [Cell 3] Model architecture and pretraining explanation for the notebook
# Assignment note: clearly explain architecture and pretraining objective.
# Source: Hugging Face model card for microsoft/Phi-3-mini-4k-instruct.
model_explanation = f"""
## Pretrained Instruction-Tuned LLM Used

Model: `{MODEL_ID}`

For the main assignment run, this script is configured for `microsoft/Phi-3-mini-4k-instruct`.
Phi-3 Mini-4K-Instruct is a 3.8B-parameter dense decoder-only Transformer language model
with a 4K-token context window. It is pretrained as an autoregressive causal language model,
meaning it learns to predict the next token from previous tokens. Its training data combines
filtered public web/code/educational text, synthetic textbook-like data, and chat-format data.
The instruction-tuned version is further aligned with supervised fine-tuning and Direct
Preference Optimization, which improves instruction following and safer assistant behavior.

In this experiment, the model is used only for inference. No gradients are computed, no model
weights are updated, and the classifier behavior comes entirely from prompting.
"""
print(model_explanation)



## Pretrained Instruction-Tuned LLM Used

Model: `HuggingFaceTB/SmolLM2-135M-Instruct`

For the main assignment run, this script is configured for `microsoft/Phi-3-mini-4k-instruct`.
Phi-3 Mini-4K-Instruct is a 3.8B-parameter dense decoder-only Transformer language model
with a 4K-token context window. It is pretrained as an autoregressive causal language model,
meaning it learns to predict the next token from previous tokens. Its training data combines
filtered public web/code/educational text, synthetic textbook-like data, and chat-format data.
The instruction-tuned version is further aligned with supervised fine-tuning and Direct
Preference Optimization, which improves instruction following and safer assistant behavior.

In this experiment, the model is used only for inference. No gradients are computed, no model
weights are updated, and the classifier behavior comes entirely from prompting.



In [32]:
# %% [Cell 4] Load tokenizer and model with optional 4-bit quantization
def load_instruction_model(model_id):
    """Load an instruction-tuned causal LM for inference only."""
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {"trust_remote_code": True}
    if USE_4BIT:
        # Recommended for GPU memory efficiency when bitsandbytes is installed.
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        model_kwargs["device_map"] = "auto"
    elif DEVICE.type == "cuda":
        model_kwargs["torch_dtype"] = torch.float16

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    if not USE_4BIT:
        model = model.to(DEVICE)
    model.eval()
    return tokenizer, model


tokenizer, model = load_instruction_model(MODEL_ID)


C:\GIU\Semster 6\Adavanced ML\Project 3\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mazen\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM2-135M-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 4322.81it/s]

In [33]:
# %% [Cell 5] Prompt templates for zero-shot, few-shot, and chain-of-thought
def format_examples(num_examples):
    """Use labeled training examples only inside the few-shot prompt."""
    lines = []
    for row in train_split.select(range(num_examples)):
        lines.append(f"Text: {row['text']}\nLabel: {label_names[row['label']]}")
    return "\n\n".join(lines)


def build_prompt(text, setting):
    label_list = ", ".join(label_names)
    base_instruction = (
        "Classify the emotion expressed in the text. "
        f"Choose exactly one label from this list: {label_list}. "
        "Return the answer in the format: Final label: <label>."
    )

    if setting == "zero_shot":
        return f"{base_instruction}\n\nText: {text}\nFinal label:"

    if setting == "few_shot_3":
        return (
            f"{base_instruction}\n\n"
            "Here are labeled examples:\n"
            f"{format_examples(3)}\n\n"
            f"Text: {text}\nFinal label:"
        )

    if setting == "few_shot_8":
        return (
            f"{base_instruction}\n\n"
            "Here are labeled examples:\n"
            f"{format_examples(8)}\n\n"
            f"Text: {text}\nFinal label:"
        )

    if setting == "chain_of_thought":
        return (
            "Classify the emotion expressed in the text. "
            f"Choose exactly one label from this list: {label_list}.\n"
            "Reason step by step about the emotional cues, then end with exactly one line "
            "in this format: Final label: <label>.\n\n"
            f"Text: {text}\nReasoning:"
        )

    raise ValueError(f"Unknown setting: {setting}")


settings = ["zero_shot", "few_shot_3", "few_shot_8", "chain_of_thought"]


In [34]:
# %% [Cell 6] Generation and label parsing
def generate_raw_output(prompt, setting):
    """Generate model text without computing gradients."""
    encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072)
    encoded = {key: value.to(model.device) for key, value in encoded.items()}
    max_tokens = COT_MAX_NEW_TOKENS if setting == "chain_of_thought" else MAX_NEW_TOKENS

    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated[0][encoded["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def parse_label(raw_output):
    """Extract the final emotion label from raw generated text."""
    normalized = raw_output.lower()
    final_match = re.search(r"final\s+label\s*:\s*(sadness|joy|love|anger|fear|surprise)", normalized)
    if final_match:
        return final_match.group(1)

    for label in label_names:
        if re.search(rf"\b{label}\b", normalized):
            return label

    # If parsing fails, use a deterministic fallback so metrics still run.
    return "sadness"


In [35]:
# %% [Cell 7] Evaluate one prompting setting
def evaluate_setting(setting):
    """Evaluate accuracy, macro F1, confusion matrix, raw examples, and inference time."""
    true_labels = []
    pred_labels = []
    examples = []

    start_time = time.time()
    for index, row in enumerate(test_split):
        prompt = build_prompt(row["text"], setting)
        raw_output = generate_raw_output(prompt, setting)
        predicted_label = parse_label(raw_output)

        true_label_id = row["label"]
        pred_label_id = label_to_id[predicted_label]
        true_labels.append(true_label_id)
        pred_labels.append(pred_label_id)

        # Show 3-5 prompt/output examples in the notebook for each setting.
        if len(examples) < 5:
            examples.append(
                {
                    "index": index,
                    "true_label": label_names[true_label_id],
                    "prompt": prompt,
                    "raw_output": raw_output,
                    "parsed_label": predicted_label,
                }
            )

        if (index + 1) % 25 == 0:
            print(f"{setting}: evaluated {index + 1}/{len(test_split)} examples")

    inference_time = time.time() - start_time
    accuracy = accuracy_score(true_labels, pred_labels)
    macro_f1 = f1_score(
        true_labels,
        pred_labels,
        labels=range(len(label_names)),
        average="macro",
        zero_division=0,
    )

    return {
        "setting": setting,
        "true_labels": true_labels,
        "pred_labels": pred_labels,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "inference_time": inference_time,
        "examples": examples,
    }


In [36]:
# %% [Cell 8] Confusion matrix plotting
def plot_confusion_matrix(result):
    cm = confusion_matrix(result["true_labels"], result["pred_labels"], labels=range(len(label_names)))
    setting = result["setting"]

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_names,
        yticklabels=label_names,
    )
    plt.title(f"LLM Prompting Confusion Matrix - {setting}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    output_path = OUTPUT_DIR / f"llm_confusion_matrix_{setting}.png"
    plt.savefig(output_path, dpi=150)
    plt.close()
    return output_path


In [37]:
# %% [Cell 9] Run all prompting settings
all_results = []
for setting in settings:
    print(f"\nEvaluating setting: {setting}")
    result = evaluate_setting(setting)
    result["confusion_matrix_path"] = str(plot_confusion_matrix(result))
    all_results.append(result)



Evaluating setting: zero_shot



Evaluating setting: few_shot_3



Evaluating setting: few_shot_8



Evaluating setting: chain_of_thought


In [38]:
# %% [Cell 10] Report metrics and show raw prompts/outputs
summary_rows = []
examples_for_export = {}

for result in all_results:
    summary_rows.append(
        {
            "setting": result["setting"],
            "accuracy": result["accuracy"],
            "macro_f1": result["macro_f1"],
            "inference_time_seconds": result["inference_time"],
            "confusion_matrix": result["confusion_matrix_path"],
        }
    )
    examples_for_export[result["setting"]] = result["examples"]

print("\nPrompting results")
print("-----------------")
for row in summary_rows:
    print(
        f"{row['setting']:<16} | "
        f"Accuracy: {row['accuracy']:.4f} | "
        f"Macro F1: {row['macro_f1']:.4f} | "
        f"Inference time: {row['inference_time_seconds']:.2f}s | "
        f"CM: {row['confusion_matrix']}"
    )

print("\nExample prompts and raw outputs")
print("-------------------------------")
for setting, examples in examples_for_export.items():
    print(f"\n### {setting}")
    for example in examples[:5]:
        print(f"\nTrue label: {example['true_label']}")
        print("Prompt:")
        print(example["prompt"])
        print("Raw output:")
        print(example["raw_output"])
        print(f"Parsed label: {example['parsed_label']}")

with open(OUTPUT_DIR / "llm_prompting_summary.json", "w", encoding="utf-8") as file:
    json.dump(summary_rows, file, indent=2)

with open(OUTPUT_DIR / "llm_prompt_examples.json", "w", encoding="utf-8") as file:
    json.dump(examples_for_export, file, indent=2)

print(f"\nSaved outputs in: {OUTPUT_DIR}")



Prompting results
-----------------
zero_shot        | Accuracy: 0.4167 | Macro F1: 0.1801 | Inference time: 42.26s | CM: llm_prompting_outputs\llm_confusion_matrix_zero_shot.png
few_shot_3       | Accuracy: 0.3333 | Macro F1: 0.1422 | Inference time: 139.91s | CM: llm_prompting_outputs\llm_confusion_matrix_few_shot_3.png
few_shot_8       | Accuracy: 0.3750 | Macro F1: 0.1821 | Inference time: 87.61s | CM: llm_prompting_outputs\llm_confusion_matrix_few_shot_8.png
chain_of_thought | Accuracy: 0.3750 | Macro F1: 0.1290 | Inference time: 222.71s | CM: llm_prompting_outputs\llm_confusion_matrix_chain_of_thought.png

Example prompts and raw outputs
-------------------------------

### zero_shot

True label: sadness
Prompt:
Classify the emotion expressed in the text. Choose exactly one label from this list: sadness, joy, love, anger, fear, surprise. Return the answer in the format: Final label: <label>.

Text: im feeling rather rotten so im not very ambitious right now
Final label:
Raw outp